# Solution: Simple PID Controller

Two independent axes, two independent controllers — see `STAGE01_FINDINGS.md` for
the full story (including an earlier wrong finding that got corrected).

- **Pitch** (`wheel_motor`): plain PD on pitch angle + pitch rate. Converges to
  under 1 degree from any single-axis disturbance up to 20 degrees tested.
- **Roll** (`reaction_wheel_motor`): plain PD on roll angle + roll rate. Also
  works very well **in isolation**.

**Known limitation, left in on purpose:** run both loops together and roll
stops recovering, even from a small disturbance — this is genuine gyroscopic
coupling (correcting pitch means accelerating/decelerating a wheel with real
angular momentum, which precesses when the chassis rotates about a different
axis), not a tuning problem. A decoupled per-axis PID structurally can't see
that coupling. That's exactly the gap PPO in stage 02 is for.


In [1]:
# Import standard libraries
import math
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer


In [2]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
WHEEL_MOTOR = "wheel_motor"
REACTION_WHEEL_MOTOR = "reaction_wheel_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"


In [3]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))


In [4]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data = mujoco.MjData(model)


In [5]:
# Get ID of actuators from MJCF names
wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, WHEEL_MOTOR)
reaction_wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, REACTION_WHEEL_MOTOR)

print(f"Wheel motor ID: {wheel_motor_id}")
print(f"Reaction wheel motor ID: {reaction_wheel_motor_id}")


Wheel motor ID: 0
Reaction wheel motor ID: 1


## Gains

Found by headless grid search across many disturbance scenarios, not
hand-guessed (see the session notes / `STAGE01_FINDINGS.md`). These are the
values that make each axis work *in isolation* — the coupling limitation
described above shows up regardless of further tuning.

In [6]:
# Pitch axis (wheel_motor)
ALPHA = 0.98    # complementary filter weight
KP = 30.0       # pitch angle
KD = 1.0        # pitch rate

# Roll axis (reaction_wheel_motor)
KP_ROLL = 10.0
KD_ROLL = 1.0

# Start tilted this many degrees in roll (about world X) instead of perfectly upright -
# useful for testing what KP_ROLL=KD_ROLL=0 actually does: from a dead-level start there's
# nothing pushing roll off zero except gyroscopic coupling from the actively-controlled
# pitch wheel, which takes several seconds to become visible. Starting pre-tilted shows the
# (lack of) roll response immediately instead of waiting for that to build up.
ROLL_START_DEG = 3.0

# When the robot has "tipped over" into an unrecoverable state
TIP_THRESHOLD = math.radians(75)

In [7]:
# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Start tilted ROLL_START_DEG in roll (rotation about world X) instead of dead-level -
# free joint qpos layout is [x, y, z, qw, qx, qy, qz]; a rotation of theta about X is
# quaternion [cos(theta/2), sin(theta/2), 0, 0]
roll_start_rad = math.radians(ROLL_START_DEG)
data.qpos[3] = math.cos(roll_start_rad / 2)
data.qpos[4] = math.sin(roll_start_rad / 2)
data.qpos[5] = 0.0
data.qpos[6] = 0.0

mujoco.mj_forward(model, data)

# Launch MuJoCo simulator and GUI
steps = 0
pitch = 0.0
roll = 0.0
tipped = False
prev_time = 0.0
with mujoco.viewer.launch_passive(model, data) as viewer:
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.08]
    viewer.cam.distance = 0.8
    viewer.cam.azimuth = 45
    viewer.cam.elevation = -25

    while viewer.is_running():
        step_start = time.time()

        # Detect MuJoCo reset (backspace) and reset filter state
        if data.time < prev_time:
            tipped = False
            # MuJoCo's own Backspace reset puts qpos back to the model's default (dead-level) -
            # re-apply the same ROLL_START_DEG tilt so every reset behaves the same way
            data.qpos[3] = math.cos(roll_start_rad / 2)
            data.qpos[4] = math.sin(roll_start_rad / 2)
            data.qpos[5] = 0.0
            data.qpos[6] = 0.0
            mujoco.mj_forward(model, data)
            accel_x, accel_y, accel_z = data.sensor(IMU_ACCEL).data
            pitch = -1 * math.atan2(accel_x, accel_z)
            roll = math.atan2(accel_y, accel_z)
        prev_time = data.time

        # Read sensors
        accel_x, accel_y, accel_z = data.sensor(IMU_ACCEL).data
        pitch_rate = data.sensor(IMU_GYRO).data[1]
        roll_rate = data.sensor(IMU_GYRO).data[0]

        # Accelerometer-derived tilt (only valid while the robot isn\'t in free-fall -
        # that\'s what the gyro half of the filter is for)
        accel_pitch = -1 * math.atan2(accel_x, accel_z)
        accel_roll = math.atan2(accel_y, accel_z)

        # Complementary filter: gyro integration for fast response, accelerometer
        # to correct long-term drift
        pitch = ALPHA * (pitch + (pitch_rate * model.opt.timestep)) + (1 - ALPHA) * accel_pitch
        roll = ALPHA * (roll + (roll_rate * model.opt.timestep)) + (1 - ALPHA) * accel_roll

        # Uncomment to use ground-truth orientation instead of the estimated pitch/roll
        # w, x, y, z = data.sensor(IMU_ORIENTATION).data
        # pitch = math.asin(max(-1.0, min(1.0, 2.0 * (w * y - z * x))))
        # roll = math.atan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))

        if abs(pitch) > TIP_THRESHOLD or abs(roll) > TIP_THRESHOLD:
            tipped = True

        if tipped:
            wheel_cmd = 0.0
            rwheel_cmd = 0.0
        else:
            wheel_cmd = clamp(KP * pitch + KD * pitch_rate)
            rwheel_cmd = clamp(KP_ROLL * roll + KD_ROLL * roll_rate)

        data.ctrl[wheel_motor_id] = wheel_cmd
        data.ctrl[reaction_wheel_motor_id] = rwheel_cmd

        mujoco.mj_step(model, data)
        viewer.sync()

        steps += 1
        if steps % PRINT_EVERY == 0:
            clear_output(wait=True)
            print(f"pitch: {math.degrees(pitch):6.2f} deg   roll: {math.degrees(roll):6.2f} deg   tipped: {tipped}")

        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)

pitch:  85.31 deg   roll: -89.99 deg   tipped: True
